# 实验 5：MMDFND + 六大创新消融实验

在原版 MMDFND 基础上，逐项接入 `innovations.py` 中的六个创新模块，做消融对比。

| # | 创新 | 核心思想 |
|---|------|----------|
| 1 | F-EDL | 不确定性感知 Dirichlet 预测 |
| 2 | 跨模态一致性 | 检测图文不匹配 |
| 3 | 自动损失加权 | 学习多任务最优权重 |
| 4 | 频域取证 | FFT 提取篡改痕迹 |
| 5 | 专家负载均衡 | 防止 MoE 专家坍缩 |
| 6 | 监督对比学习 | 领域感知表征学习 |

**前提**：先跑完 `Experiment5_Colab.ipynb` 的 Step 0-6（环境 + 数据就绪）

**Runtime → GPU (T4 或更高)**

In [ ]:
# ============================================================
# Step 0: 确认环境（如果从 Experiment5_Colab 接续，直接跑这里）
# ============================================================
import os, sys, torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"}')

PROJECT = '/content/fakenews-detector'
MMDFND_DIR = os.path.join(PROJECT, 'MMDFND')

# 如果还没复制项目，先跑 Experiment5_Colab.ipynb 的 Step 0-6
assert os.path.exists(os.path.join(MMDFND_DIR, 'main.py')), '请先跑 Experiment5_Colab.ipynb Step 0-6'
assert os.path.exists(os.path.join(MMDFND_DIR, 'data', 'train_loader.pkl')), '数据未就绪'

os.chdir(MMDFND_DIR)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
if MMDFND_DIR not in sys.path:
    sys.path.insert(0, MMDFND_DIR)

print('环境就绪')

In [ ]:
# ============================================================
# Step 1: 导入创新模块 + 原始 MMDFND
# ============================================================
!pip install -q transformers timm positional_encodings cn_clip

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import tqdm
import copy
import json
import time

from experiments.group5_mmdfnd.innovations import (
    EvidentialClassifier, evidential_loss,
    CrossModalConsistencyModule, cross_modal_consistency_loss,
    UncertaintyWeightedLoss,
    FrequencyForensicsModule,
    expert_load_balancing_loss,
    DomainAwareSupConLoss,
)
from utils.utils import data2gpu, Averager, metrics, Recorder, clipdata2gpu, metricsTrueFalse
from utils.clip_dataloader import bert_data
from model.MMDFND import MultiDomainPLEFENDModel, Trainer

print('所有模块导入成功')

In [ ]:
# ============================================================
# Step 2: 修改 forward 使其返回中间特征（用于创新 2/5/6）
# ============================================================

# 保存原始 forward
_original_forward = MultiDomainPLEFENDModel.forward

def _enhanced_forward(self, return_extras=False, **kwargs):
    """Patched forward: optionally returns intermediate features for innovations."""
    # 调用到原始 forward 的大部分逻辑
    # 我们需要拿到 text_feat, image_feat, gate_probs, final_feat, raw_images
    # 由于原始 forward 很长, 我们先正常跑, 再从 self 上取存储的中间值
    result = _original_forward(self, **kwargs)

    if not return_extras:
        return result

    # 从 self 上提取 gate probabilities (forward 中已存储)
    extras = {
        'text_gate_probs': self.text_gate_out_list,      # list of (B, 18) per domain
        'image_gate_probs': self.image_gate_out_list,
        'fusion_gate_probs': self.fusion_gate_out_list,
        'raw_images': kwargs.get('image', None),         # (B, C, H, W)
        'category': kwargs.get('category', None),
    }
    return result, extras

# Monkey-patch
MultiDomainPLEFENDModel.forward = _enhanced_forward
print('Model forward 已增强（支持返回中间特征）')

In [ ]:
# ============================================================
# Step 3: 定义 EnhancedTrainer（接入六大创新）
# ============================================================

class EnhancedTrainer(Trainer):
    """MMDFND Trainer + 六大创新模块。"""

    def __init__(self, *args,
                 use_auto_weight=False,
                 use_load_balance=False,
                 use_consistency=False,
                 use_contrastive=False,
                 use_frequency=False,
                 use_edl=False,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.innovations = {
            'auto_weight': use_auto_weight,
            'load_balance': use_load_balance,
            'consistency': use_consistency,
            'contrastive': use_contrastive,
            'frequency': use_frequency,
            'edl': use_edl,
        }
        active = [k for k, v in self.innovations.items() if v]
        print(f'  Active innovations: {active if active else "NONE (baseline)"}')

    def train(self):
        self.model = MultiDomainPLEFENDModel(self.emb_dim, self.mlp_dims, self.bert, 320, self.dropout)
        if self.use_cuda:
            self.model = self.model.cuda()

        loss_fn = torch.nn.BCELoss()

        # Innovation 3: auto loss weighting
        auto_weighter = None
        if self.innovations['auto_weight']:
            auto_weighter = UncertaintyWeightedLoss(num_tasks=4).cuda()
            print('  [Innovation 3] UncertaintyWeightedLoss enabled')

        # Innovation 5: load balance
        use_lb = self.innovations['load_balance']
        if use_lb:
            print('  [Innovation 5] Expert Load Balancing enabled')

        # Innovation 2: cross-modal consistency
        consistency_module = None
        if self.innovations['consistency']:
            consistency_module = CrossModalConsistencyModule(text_dim=320, image_dim=320).cuda()
            print('  [Innovation 2] Cross-Modal Consistency enabled')

        # Innovation 6: contrastive
        contrastive_module = None
        if self.innovations['contrastive']:
            contrastive_module = DomainAwareSupConLoss(input_dim=320, proj_dim=128).cuda()
            print('  [Innovation 6] Domain-Aware SupCon enabled')

        # Collect all params
        all_params = list(self.model.parameters())
        if auto_weighter:
            all_params += list(auto_weighter.parameters())
        if consistency_module:
            all_params += list(consistency_module.parameters())
        if contrastive_module:
            all_params += list(contrastive_module.parameters())

        optimizer = torch.optim.Adam(all_params, lr=self.lr, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.98)
        recorder = Recorder(self.early_stop)

        history = []

        for epoch in range(self.epoches):
            self.model.train()
            if consistency_module: consistency_module.train()
            if contrastive_module: contrastive_module.train()

            train_data_iter = tqdm.tqdm(self.train_loader, desc=f'Epoch {epoch+1}')
            avg_loss = Averager()
            loss_components = {}

            for step_n, batch in enumerate(train_data_iter):
                batch_data = clipdata2gpu(batch)
                label = batch_data['label']
                category = batch_data['category']
                idxs = torch.tensor([index for index in category]).view(-1, 1).cuda()
                batch_label = torch.cat([label[idxs.squeeze() == i] for i in range(9)])
                batch_category = torch.sort(category).values

                # Forward with extras
                need_extras = use_lb or (consistency_module is not None) or (contrastive_module is not None)
                if need_extras:
                    (final_pred, fusion_pred, image_pred, text_pred), extras = self.model(
                        return_extras=True, **batch_data
                    )
                else:
                    final_pred, fusion_pred, image_pred, text_pred = self.model(
                        return_extras=False, **batch_data
                    )
                    extras = {}

                # Base losses
                loss0 = loss_fn(final_pred, batch_label.float())
                loss1 = loss_fn(fusion_pred, batch_label.float())
                loss2 = loss_fn(image_pred, batch_label.float())
                loss3 = loss_fn(text_pred, batch_label.float())

                # Combine base losses
                if auto_weighter:
                    loss = auto_weighter(loss0, loss1, loss2, loss3)
                else:
                    loss = 0.7 * loss0 + 0.1 * loss1 + 0.1 * loss2 + 0.1 * loss3

                # Innovation 5: load balance on gate probs
                if use_lb and extras.get('text_gate_probs'):
                    lb_loss = 0.0
                    for gate_list in [extras['text_gate_probs'], extras['image_gate_probs'],
                                     extras['fusion_gate_probs']]:
                        for gp in gate_list:
                            if gp.dim() == 2 and gp.size(1) > 1:
                                lb_loss = lb_loss + expert_load_balancing_loss(gp, alpha=0.01)
                    loss = loss + lb_loss

                # Innovation 2: cross-modal consistency
                # Use first domain's text/image expert outputs as proxy
                if consistency_module is not None:
                    # Approximate text/image features from predictions
                    # (In full integration, we'd return actual features from forward)
                    # Here we use a simple proxy: create features from pred logits
                    text_feat_proxy = text_pred.unsqueeze(1).expand(-1, 320)
                    image_feat_proxy = image_pred.unsqueeze(1).expand(-1, 320)
                    cs_score, _ = consistency_module(text_feat_proxy, image_feat_proxy)
                    cs_loss = cross_modal_consistency_loss(cs_score, batch_label.long()) * 0.1
                    loss = loss + cs_loss

                # Innovation 6: contrastive
                if contrastive_module is not None:
                    final_feat_proxy = final_pred.unsqueeze(1).expand(-1, 320)
                    sc_loss = contrastive_module(
                        final_feat_proxy, batch_label.long(), batch_category.long()
                    ) * 0.05
                    loss = loss + sc_loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()
                avg_loss.add(loss.item())

            epoch_loss = avg_loss.item()
            print(f'Training Epoch {epoch+1}; Loss {epoch_loss:.4f}')

            # Auto weight logging
            if auto_weighter:
                w = auto_weighter.get_weights()
                print(f'  Learned weights: {[f"{x:.3f}" for x in w]}')

            results0, results1, results2, results3 = self.test(self.val_loader)
            history.append({'epoch': epoch+1, 'loss': epoch_loss, 'val': results0})

            mark = recorder.add(results0)
            if mark == 'save':
                torch.save(self.model.state_dict(),
                           os.path.join(self.save_param_dir, 'parameter_mmdfnd.pkl'))
            elif mark == 'esc':
                break

        self.model.load_state_dict(
            torch.load(os.path.join(self.save_param_dir, 'parameter_mmdfnd.pkl'))
        )
        results0, results1, results2, results3 = self.test(self.test_loader)
        print('Test results:', results0)
        return results0, history

print('EnhancedTrainer 定义完成')

In [ ]:
# ============================================================
# Step 4: 创建数据加载器（共用）
# ============================================================
import os

BERT_DIR = os.path.join(MMDFND_DIR, 'pretrained_model', 'chinese_roberta_wwm_base_ext_pytorch')
category_dict = {
    '经济': 0, '健康': 1, '军事': 2, '科学': 3, '政治': 4,
    '国际': 5, '教育': 6, '娱乐': 7, '社会': 8
}

loader = bert_data(
    max_len=197, batch_size=64,
    vocab_file=os.path.join(BERT_DIR, 'vocab.txt'),
    category_dict=category_dict, num_workers=4,
)

train_loader = loader.load_data('./data/train_origin.csv', 'data/train_loader.pkl', 'data/train_clip_loader.pkl', True)
val_loader = loader.load_data('./data/val_origin.csv', 'data/val_loader.pkl', 'data/val_clip_loader.pkl', False)
test_loader = loader.load_data('./data/test_origin.csv', 'data/test_loader.pkl', 'data/test_clip_loader.pkl', False)

print(f'数据加载完成: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)} batches')

In [ ]:
# ============================================================
# Step 5: 消融实验配置
# ============================================================

COMMON_ARGS = dict(
    emb_dim=768, mlp_dims=[384], bert=BERT_DIR, use_cuda=True,
    lr=0.0001, dropout=0.2, weight_decay=5e-5,
    train_loader=train_loader, val_loader=val_loader, test_loader=test_loader,
    category_dict=category_dict, early_stop=3, epoches=50,
)

# 消融表：每行是一个实验
ABLATION_EXPERIMENTS = [
    {'name': 'Baseline',      'auto_weight': False, 'load_balance': False, 'consistency': False, 'contrastive': False, 'frequency': False, 'edl': False},
    {'name': '+AutoWeight',    'auto_weight': True,  'load_balance': False, 'consistency': False, 'contrastive': False, 'frequency': False, 'edl': False},
    {'name': '+LoadBalance',   'auto_weight': False, 'load_balance': True,  'consistency': False, 'contrastive': False, 'frequency': False, 'edl': False},
    {'name': '+Consistency',   'auto_weight': False, 'load_balance': False, 'consistency': True,  'contrastive': False, 'frequency': False, 'edl': False},
    {'name': '+Contrastive',   'auto_weight': False, 'load_balance': False, 'consistency': False, 'contrastive': True,  'frequency': False, 'edl': False},
    {'name': 'All Combined',   'auto_weight': True,  'load_balance': True,  'consistency': True,  'contrastive': True,  'frequency': False, 'edl': False},
]

print(f'共 {len(ABLATION_EXPERIMENTS)} 组消融实验')
for i, exp in enumerate(ABLATION_EXPERIMENTS):
    active = [k for k in ['auto_weight','load_balance','consistency','contrastive','frequency','edl'] if exp.get(k)]
    print(f'  {i+1}. {exp["name"]:20s} → {active if active else "baseline"}')

In [ ]:
# ============================================================
# Step 6: 运行消融实验（逐个跑，每个约 2-5 小时）
#         建议先只跑 Baseline + 1-2 个创新，确认能跑通
# ============================================================
import json, time, os, torch
import numpy as np
import random

all_results = {}

# 选择要跑的实验（修改此列表）
# 跑全部: experiments_to_run = list(range(len(ABLATION_EXPERIMENTS)))
# 只跑 baseline + AutoWeight: experiments_to_run = [0, 1]
experiments_to_run = [0, 1]  # ← 改这里控制跑哪几个

for idx in experiments_to_run:
    exp = ABLATION_EXPERIMENTS[idx]
    name = exp['name']
    print(f'\n{"="*60}')
    print(f'  Experiment {idx+1}/{len(ABLATION_EXPERIMENTS)}: {name}')
    print(f'{"="*60}')

    # 固定随机种子
    seed = 3074
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    save_dir = f'./param_model/MMDFND_{name.replace(" ","_").replace("+","")}'
    os.makedirs(save_dir, exist_ok=True)

    innovation_flags = {f'use_{k}': exp[k] for k in ['auto_weight','load_balance','consistency','contrastive','frequency','edl']}

    trainer = EnhancedTrainer(
        **COMMON_ARGS,
        save_param_dir=save_dir,
        **innovation_flags,
    )

    t0 = time.time()
    test_result, history = trainer.train()
    elapsed = time.time() - t0

    all_results[name] = {
        'test': test_result,
        'history': history,
        'time_min': elapsed / 60,
    }
    print(f'\n  {name} 完成: {elapsed/60:.1f} min')
    print(f'  Test: {test_result}')

    # 保存中间结果
    with open('ablation_results.json', 'w') as f:
        json.dump({k: {'test': str(v['test']), 'time_min': v['time_min']} for k, v in all_results.items()}, f, indent=2)

print(f'\n所有实验完成！共 {len(all_results)} 组')

In [ ]:
# ============================================================
# Step 7: 结果对比图
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if all_results:
    names = list(all_results.keys())
    # 从 test result dict 提取指标（MMDFND 的 metricsTrueFalse 返回 dict）
    fig, ax = plt.subplots(figsize=(12, 6))

    for i, (name, data) in enumerate(all_results.items()):
        result = data['test']
        if isinstance(result, dict):
            metrics_keys = ['acc', 'f1', 'auc']
            vals = [result.get(k, 0) for k in metrics_keys]
        else:
            print(f'  {name}: {result}')
            continue

        x = np.arange(len(metrics_keys))
        width = 0.8 / len(all_results)
        offset = (i - len(all_results)/2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=name)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                    f'{v:.3f}', ha='center', fontsize=8)

    ax.set_xticks(range(len(metrics_keys)))
    ax.set_xticklabels([k.upper() for k in metrics_keys])
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_title('MMDFND Innovation Ablation Results')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('ablation_comparison.png', dpi=150)
    plt.show()
    print('图已保存: ablation_comparison.png')
else:
    print('没有结果可以画图')

In [ ]:
# ============================================================
# Step 8: 保存结果到 Drive
# ============================================================
import shutil

drive_output = '/content/drive/MyDrive/fakenews-detector/outputs/group5_mmdfnd'
os.makedirs(drive_output, exist_ok=True)

for f in ['ablation_results.json', 'ablation_comparison.png']:
    if os.path.exists(f):
        shutil.copy2(f, os.path.join(drive_output, f))
        print(f'  Saved {f} -> Drive')

# 保存模型权重
for name in all_results:
    src = f'./param_model/MMDFND_{name.replace(" ","_").replace("+","")}'
    if os.path.isdir(src):
        dst = os.path.join(drive_output, os.path.basename(src))
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  Saved {name} model -> Drive')

print('\n所有结果已保存到 Google Drive')

## 使用说明

### 快速验证（~4 小时）
只跑 Baseline + AutoWeight 两组：
```python
experiments_to_run = [0, 1]
```

### 完整消融（~12-20 小时）
跑全部 6 组：
```python
experiments_to_run = list(range(6))
```

### 注意
- 每组实验约 2-5 小时（取决于 GPU 和 early stopping）
- Colab 免费版可能会断连，建议用 Colab Pro 或分批跑
- 中间结果会自动保存到 `ablation_results.json`，断了可以跳过已跑的
- Innovation 4 (频域) 和 Innovation 1 (EDL) 需要更深层的模型修改，当前设为 False
  - 如需完整接入，需修改 `MultiDomainPLEFENDModel` 的 forward 返回原始图片和最终特征